# MOE Medical Vision - Post Reboot Recovery

Ejecutar despues de reiniciar el pod.

In [1]:
# Verificar espacio
import subprocess
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(result.stdout)

# Verificar datasets
from pathlib import Path
RAW = Path('/workspace/moe_medical_vision/data/raw')

datasets = ['nih', 'isic', 'osteoporosis', 'luna16', 'pancreatic']
for ds in datasets:
    folder = RAW / ds
    if folder.exists():
        size = subprocess.run(['du', '-sh', str(folder)], capture_output=True, text=True).stdout.strip()
        count = sum(1 for _ in folder.rglob('*') if _.is_file())
        print(f'{ds}: {count} archivos, {size}')
    else:
        print(f'{ds}: No existe')

Filesystem      Size  Used Avail Use% Mounted on
overlay          30G   30G   23M 100% /

nih: 0 archivos, 512	/workspace/moe_medical_vision/data/raw/nih
isic: 0 archivos, 512	/workspace/moe_medical_vision/data/raw/isic
osteoporosis: 12719 archivos, 10G	/workspace/moe_medical_vision/data/raw/osteoporosis
luna16: 1794 archivos, 669M	/workspace/moe_medical_vision/data/raw/luna16
pancreatic: 0 archivos, 512	/workspace/moe_medical_vision/data/raw/pancreatic


## 1. Descargar NIH (~45GB)

In [ ]:
import kagglehub
from pathlib import Path
import shutil

NIH_DIR = Path('/workspace/moe_medical_vision/data/raw/nih')

# Limpiar si existe
import shutil
if NIH_DIR.exists():
    shutil.rmtree(NIH_DIR)
NIH_DIR.mkdir(parents=True)

print('Descargando NIH (~45GB)...')
path = kagglehub.dataset_download('nih-chest-xrays/data')
print(f'Descarga: {path}')

# Mover archivos
for item in Path(path).iterdir():
    dst = NIH_DIR / item.name
    if dst.exists(): dst.unlink()
    shutil.move(str(item), str(dst))

Path(path).rmdir()
print('NIH listo!')

## 2. Descargar ISIC (~5GB)

In [ ]:
import subprocess
from pathlib import Path

ISIC_DIR = Path('/workspace/moe_medical_vision/data/raw/isic')
ISIC_DIR.mkdir(parents=True, exist_ok=True)

print('Descargando ISIC 2019 (~5GB)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'andrewmvd/isic-2019', '-p', str(ISIC_DIR), '--unzip'],
    capture_output=True, text=True, timeout=7200
)
if result.returncode == 0:
    print('ISIC 2019 listo!')
else:
    print(f'ERROR: {result.stderr[-500:]}')

## 3. Descargar Pancreatic (~46GB)

In [ ]:
import subprocess
import zipfile
from pathlib import Path

DEST = Path('/workspace/moe_medical_vision/data/raw/pancreatic')
DEST.mkdir(parents=True, exist_ok=True)
ZIP = DEST / 'batch_1.zip'

print('Descargando Pancreatic Cancer (~46GB desde Zenodo)...')
result = subprocess.run(
    ['wget', '--progress=bar:force', '-O', str(ZIP),
     'https://zenodo.org/records/13715870/files/batch_1.zip'],
    timeout=28800
)

if result.returncode == 0 and ZIP.exists():
    print('Descarga completa. Extrayendo...')
    with zipfile.ZipFile(ZIP, 'r') as z:
        z.extractall(DEST)
    ZIP.unlink()  # Liberar ~46GB
    print('Pancreatic Cancer listo!')
else:
    print('ERROR en descarga')

## 4. Reporte Final

In [ ]:
import subprocess
from pathlib import Path

def get_size(path):
    result = subprocess.run(['du', '-sh', str(path)], capture_output=True, text=True)
    return result.stdout.split()[0] if result.returncode == 0 else '0'

def count_files(path):
    return sum(1 for f in Path(path).rglob('*') if f.is_file())

print('='*60)
print(' REPORTE DE INTEGRIDAD - MOE Medical Vision')
print('='*60)

datasets = ['nih', 'isic', 'osteoporosis', 'luna16', 'pancreatic']
RAW = Path('/workspace/moe_medical_vision/data/raw')
total = 0

for ds in datasets:
    folder = RAW / ds
    if folder.exists():
        n = count_files(folder)
        s = get_size(folder)
        print(f'{ds:<15} {n:>8} archivos  {s:>8}')
    else:
        print(f'{ds:<15} {'0':>8} archivos  {'0':>8}')

print('='*60)
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(result.stdout)